In [ ]:
!dd if=/dev/urandom of=sample_file.bin bs=128 count=1
!openssl sha256 sample_file.bin

In [ ]:
import mmap

import ctypes
import json
import hashlib
import numpy as np
import random
import time

with open('sample_file.bin', 'r+b') as f:
    with mmap.mmap(f.fileno(), length=0, access=mmap.ACCESS_WRITE) as mm:
        # Memory mapped file mm behaves like a bytearray
        sha256_hash = hashlib.sha256(mm).hexdigest()
        print(f'Hash before sort: {sha256_hash}')

        # Treat memory mapped file bytes as integers array in memory.
        # Sorting them.
        int_array = np.frombuffer(mm, dtype=np.uint32)
        print('Bytes as array of garbage 32 unsigned integers')
        print(int_array[:32]) # Unsorted
        int_array.sort()
        print('Bytes as array of sorted 32 unsigned integers')
        print(int_array[:32]) # Sorted
        mm.flush()
        
        # numpy array must be closed before memory mapped file.
        # BufferError: cannot close exported pointers exist
        del int_array

        sha256_hash = hashlib.sha256(mm).hexdigest()
        print(f'Hash after sort: {sha256_hash}')

In [ ]:
with open('sample_file.bin', 'rb') as f:
    file_content = f.read()
    sha256_hash = hashlib.sha256(file_content).hexdigest()
    print(f'Checking that file content changed indeed: {sha256_hash}')
    
    with mmap.mmap(f.fileno(), length=0, access=mmap.ACCESS_READ) as mm:
        sha256_hash = hashlib.sha256(mm).hexdigest()
        print(f'Hash computed from memory mapped file: {sha256_hash}')

        try:
            mm[0] = b'x'
            assert('Cannot write to read-only mmap')
        except TypeError as e:
            #TypeError: mmap can't modify a readonly memory map.
            print('Exception as expected: ', e)

In [ ]:
with open('sample_file.bin', 'r+b') as f:
    with mmap.mmap(f.fileno(), length=0, access=mmap.ACCESS_WRITE) as mm:
        # Memory mapped file mm behaves like a bytearray
        sha256_hash = hashlib.sha256(mm).hexdigest()
        print(sha256_hash)

        # Treating buffer as signed integers (~ half of them will be negative)
        IntArrayType = ctypes.c_int * (len(mm) // ctypes.sizeof(ctypes.c_int))
        int_array = IntArrayType.from_buffer(mm)
        
        j = json.dumps({'numbers' : [n for n in int_array]}, indent=4).encode('utf-8')
        print('Previously sorted 32 bit unsigned integers treated as signed')
        print(j)

        # Cannot close the memory mapped file while the integer array still holds a reference.
        # BufferError: cannot close exported pointers exist
        del int_array

    # Deceptively called function "truncate" works just fine to grow the file size forward. 
    f.truncate(len(j))

    with mmap.mmap(f.fileno(), length=0, access=mmap.ACCESS_WRITE) as mm:
        mm[0:] = j
        mm.flush()

In [ ]:
with open('sample_file.bin', 'r') as f:
    j = json.load(f)
    print('Reading presumably json file back')
    print(j)

In [ ]:
import fcntl
import struct
import subprocess
import sys

# A tiny C library with one function. It must not need anything from the C runtime, see below.
c_source = r'''
int hello_from_mmap(void) {
    int answer = 0;
    for (int i = 2, end = 14; i < end; i += 2) {
        answer += i;
    }
    return answer; // 42
}
'''
with open('hello_mmap.c', 'w') as f:
    f.write(c_source)

# Compile a position independent shared library: ELF on Linux, Mach-O on macOS, where the
# linker also signs it, since Apple silicon only runs signed code.
macos = sys.platform == 'darwin'
library = './libhello_mmap.dylib' if macos else './libhello_mmap.so'
subprocess.run(['cc', '-shared', '-fPIC', '-O2', '-o', library, 'hello_mmap.c'], check=True)

def run(*args):
    return subprocess.run(args, capture_output=True, check=True, text=True).stdout

def get_function_file_offset_elf(library, function_name):
    # nm prints the function's virtual address, where it lands once the loader maps the library:
    # 0000000000000580 T hello_from_mmap
    address = next(int(line.split()[0], 16) for line in run('nm', '-D', library).splitlines()
                   if line.split()[-1] == function_name)
    # A plain file mapping lays the bytes out as in the file, so turn the address into a file
    # offset through the .text section header: [ 9] .text PROGBITS 0000000000000480 000480 ...
    text = next(line.split(']')[1].split() for line in run('readelf', '-SW', library).splitlines()
                if '] .text ' in line)
    section_address, section_offset = int(text[2], 16), int(text[3], 16)
    return address - section_address + section_offset

def macho_load_commands(data):
    # mach_header_64: magic, cputype, cpusubtype, filetype, ncmds, sizeofcmds, flags, reserved;
    # then ncmds load commands, each starting with its type and size
    magic, _, _, _, ncmds, _, _, _ = struct.unpack_from('<8I', data)
    assert magic == 0xfeedfacf, 'not a 64-bit Mach-O file'
    position = 32
    for _ in range(ncmds):
        command, size = struct.unpack_from('<2I', data, position)
        yield command, position
        position += size

def get_function_file_offset_macho(library, function_name):
    # C symbols carry a leading underscore in Mach-O: 0000000000000398 T _hello_from_mmap
    address = next(int(line.split()[0], 16) for line in run('nm', '-U', library).splitlines()
                   if line.split()[-1] == '_' + function_name)
    # The segment holding the address (__TEXT) says where its bytes lie in the file
    with open(library, 'rb') as f:
        data = f.read()
    for command, position in macho_load_commands(data):
        if command == 0x19:  # LC_SEGMENT_64: cmd, cmdsize, segname[16], vmaddr, vmsize, fileoff, ...
            vmaddr, vmsize, fileoff = struct.unpack_from('<3Q', data, position + 24)
            if vmaddr <= address < vmaddr + vmsize:
                return address - vmaddr + fileoff
    raise LookupError(f'{function_name} lies in no segment')

def register_code_signature(f):
    # macOS only runs code from pages whose signature the kernel knows. dlopen registers a
    # library's signature before mapping it; a plain mmap must do the same, with
    # fcntl(F_ADDFILESIGS_RETURN), or the process is killed on its first instruction there.
    data = f.read()
    f.seek(0)
    for command, position in macho_load_commands(data):
        if command == 0x1d:  # LC_CODE_SIGNATURE: cmd, cmdsize, dataoff, datasize
            dataoff, datasize = struct.unpack_from('<2I', data, position + 8)
            # fsignatures_t: fs_file_start, fs_blob_start (a file offset here), fs_blob_size, ...
            F_ADDFILESIGS_RETURN = 97
            signatures = struct.pack('@qQQQ20si', 0, dataoff, datasize, 0, b'', 0)
            fcntl.fcntl(f.fileno(), F_ADDFILESIGS_RETURN, signatures)
            return
    raise LookupError('the library is not signed')

# Open the library with mmap, the way the dynamic loader itself maps shared libraries. A plain
# file mapping is not relocated, so its bytes can be read (here the file format's magic number).
# dlopen normally mmaps and relocates the library's segments
# but we recklessly call a function from the uninitialized library anyway.
with open(library, 'rb') as f:
    if macos:
        register_code_signature(f)
    # Read and execute, never write: macOS refuses memory that is writable and executable at once
    # (PermissionError: Operation not permitted). MAP_PRIVATE: nothing reaches the file on disk.
    with mmap.mmap(f.fileno(), length=0, flags=mmap.MAP_PRIVATE,
                   prot=mmap.PROT_READ | mmap.PROT_EXEC) as so:
        print(f'Mapped {len(so)} bytes of {library}, magic number {bytes(so[:4]).hex()} '
              f'({"Mach-O 64-bit" if macos else "ELF"})')

        # ctypes.c_char.from_buffer(so) needs a writable buffer (TypeError: underlying buffer is
        # not writable), but a read-only numpy view still tells where its data is.
        view = np.frombuffer(so, dtype=np.uint8)
        mmap_address = view.ctypes.data
        print(f'mmap_address: 0x{mmap_address:x}')

        # The view must go before the mapping closes (BufferError: cannot close exported pointers exist)
        del view

        offset = (get_function_file_offset_macho if macos else get_function_file_offset_elf)(library, 'hello_from_mmap')
        address = mmap_address + offset
        print(f'hello_from_mmap at file offset 0x{offset:x}, address 0x{address:x} directly from memory mapped file')

        # DANGER! Reckless call from the uninitialized library that would result in a crash
        # if that function needed anything from C runtime or accessed global variables.
        hello_from_mmap = ctypes.CFUNCTYPE(ctypes.c_int)(address)
        result = hello_from_mmap()
        print(f'hello_from_mmap() returned {result}')

lib = ctypes.CDLL(library)

# The symbol's runtime address, as a plain integer
address = ctypes.cast(lib.hello_from_mmap, ctypes.c_void_p).value
print(f'hello_from_mmap at 0x{address:x} via ctypes.CDLL')

# Build a callable from that raw address and call it by pointer: int hello_from_mmap(void)
hello_from_mmap = ctypes.CFUNCTYPE(ctypes.c_int)(address)
result = hello_from_mmap()
print(f'hello_from_mmap() returned {result}')